03 - Baseline Model

Goal of this notebook:
- load the processed train / validation / test datasets
- build a small leakage-safe feature set
- train a Logistic Regression baseline
- evaluate with metrics that matter for imbalanced fraud data
- choose a decision threshold using validation data

This notebook is intentionally simple and learning-first.

# Importing the libraries and Setup Config

In [1]:
from pathlib import Path
import pandas as pd
import os
import numpy as np
#ColumnTransformer selects specific columns, applies a transformer to each group, combines the results into one feature matrix
from sklearn.compose import ColumnTransformer # why this?
from sklearn.impute import SimpleImputer # for handling missing values, mean imputation and median imputation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline # to chain multiple data-processing and modeling steps into a single object
# One‑hot encoding is a technique used in data science to convert categorical variables into numerical form so machine‑learning models can use them
# StandardScaler is a feature‑scaling tool in scikit‑learn that standardizes numerical data so each feature has Mean = 0 and Standard deviation = 1
from sklearn.preprocessing import OneHotEncoder, StandardScaler #why this?

In [2]:
PROJECT_DIR = Path("..").resolve()
DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed"

TRAIN_PATH =  PROCESSED_DIR / "train.parquet"
VAL_PATH = PROCESSED_DIR / "val.parquet"
TEST_PATH = PROCESSED_DIR / "test.parquet"

print(F"PROJECT DIRECTORY : {PROJECT_DIR} | EXISTS : {PROJECT_DIR.exists()}")
print(F"TRAIN PATH : {PROJECT_DIR} | EXISTS : {TRAIN_PATH.exists()}")
print(F"VALIDATION PATH : {PROJECT_DIR} | EXISTS : {VAL_PATH.exists()}")
print(F"TEST PATH : {PROJECT_DIR} | EXISTS : {TEST_PATH.exists()}")


PROJECT DIRECTORY : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True
TRAIN PATH : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True
VALIDATION PATH : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True
TEST PATH : C:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection | EXISTS : True


# Data setup

In [3]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("TRAIN SHAPE :", train_df.shape )
print("VALIDATION SHAPE:", val_df.shape)
print("TEST SHAPE :", test_df.shape )


TRAIN SHAPE : (3886017, 12)
VALIDATION SHAPE: (896971, 12)
TEST SHAPE : (295357, 12)


In [4]:
train_df.columns

Index(['transaction_id', 'transaction_timestamp', 'from_bank_id',
       'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received',
       'receiving_currency', 'amount_paid', 'payment_currency',
       'payment_format', 'is_laundering'],
      dtype='str')

# Helper Functions

## EDA

In [5]:

def target_summary(name, df):
    count = df["is_laundering"].value_counts(dropna=False).sort_index().to_frame("count")
    count["pct"] = (count["count"]/len(df) *100).round(4)
    print("-"*40)
    print(name)
    print("Summary: \n",count)
    print("-"*40)
    

## Feature Engineering

In [6]:
def add_basic_features(df: pd.DataFrame) -> (pd.DataFrame):
    df=df.copy()

    # timestamp is datetime
    df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"], errors="coerce")

    # hour, day and is_weekend from timestamp
    df["hour_of_day"] = df["transaction_timestamp"].dt.hour
    df["day_of_week"] = df["transaction_timestamp"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

    #converting amount_received and ammount_paid into numeric
    df["amount_received"] =  pd.to_numeric(df["amount_received"],errors="coerce")
    df["amount_paid"] =  pd.to_numeric(df["amount_paid"],errors="coerce")

    # same_currency flag if receiving and sending payment currency is same
    df["same_currency_flag"] = (df["receiving_currency"].astype(str) == df["payment_currency"].astype(str)).astype(int)
    

    # same_bank flag if receiving and sending payment currbank is same
    df["same_bank_flag"] = (df["from_bank_id"].astype(str) == df["to_bank_id"].astype(str)).astype(int)

     # avoid log(0) - because log(0) is undefined and using np.log1p makes log(0) -> log(1) because [log(1+x)]. clip(lower = 0) makes any value below 0 is set to 0
    df["log_amount_received"] = np.log1p(df["amount_received"].clip(lower=0))
    df["log_amount_paid"] = np.log1p(df["amount_paid"].clip(lower=0))                        

    return df


# EDA

In [7]:
target_summary("Trainjng Data",train_df)

----------------------------------------
Trainjng Data
Summary: 
                  count      pct
is_laundering                  
0              3882877  99.9192
1                 3140   0.0808
----------------------------------------


In [8]:
target_summary("Validation Data",val_df)

----------------------------------------
Validation Data
Summary: 
                 count      pct
is_laundering                 
0              896093  99.9021
1                 878   0.0979
----------------------------------------


In [9]:
target_summary("Test Data",test_df)

----------------------------------------
Test Data
Summary: 
                 count      pct
is_laundering                 
0              294198  99.6076
1                1159   0.3924
----------------------------------------


## Feature engineering

Selected baseline features include:
- raw amount columns
- log amounts
- payment format
- hour of day
- day of week
- weekend flag
- same-currency flag
- same-bank vs cross-bank flag

I will keep the baseline small and row-level only.

In [10]:
train_feat = add_basic_features(train_df)
val_feat = add_basic_features(val_df)
test_feat = add_basic_features(test_df)

print("-"*40)
print("Train data - features")
print(train_feat.info())
print("-"*40)
print("Validation data - features")
print(val_feat.info())
print("-"*40)
print("Test data - features")
print(test_feat.info())
print("-"*40)

----------------------------------------
Train data - features
<class 'pandas.DataFrame'>
RangeIndex: 3886017 entries, 0 to 3886016
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   transaction_id         str           
 1   transaction_timestamp  datetime64[us]
 2   from_bank_id           str           
 3   from_account_id        str           
 4   to_bank_id             str           
 5   to_account_id          str           
 6   amount_received        float64       
 7   receiving_currency     str           
 8   amount_paid            float64       
 9   payment_currency       str           
 10  payment_format         str           
 11  is_laundering          int64         
 12  hour_of_day            int32         
 13  day_of_week            int32         
 14  is_weekend             int64         
 15  same_currency_flag     int64         
 16  same_bank_flag         int64         
 17  log_amount_re

In [11]:
train_feat.columns

Index(['transaction_id', 'transaction_timestamp', 'from_bank_id',
       'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received',
       'receiving_currency', 'amount_paid', 'payment_currency',
       'payment_format', 'is_laundering', 'hour_of_day', 'day_of_week',
       'is_weekend', 'same_currency_flag', 'same_bank_flag',
       'log_amount_received', 'log_amount_paid'],
      dtype='str')

## Baseline features

In [12]:
feature_cols=[
    'amount_received',
    'amount_paid',
    'receiving_currency',
    'payment_currency',
    'payment_format',
    'hour_of_day', 
    'day_of_week',
    'is_weekend', 
    'same_currency_flag', 
    'same_bank_flag',
    'log_amount_received', 
    'log_amount_paid'
    
]

target_col = 'is_laundering'


In [13]:
numeric_cols = [
    'amount_received',
    'amount_paid',
    'log_amount_received', 
    'log_amount_paid',
    'hour_of_day', 
    'day_of_week',
    'is_weekend', 
    'same_currency_flag', 
    'same_bank_flag'
]

categorical_cols= [
    'payment_format',
    'receiving_currency',
    'payment_currency'

]

In [14]:
X_train = train_feat[feature_cols]
Y_train = train_feat[target_col]

X_val = val_feat[feature_cols]
Y_val = val_feat[target_col]

X_test = test_feat[feature_cols]
Y_test = test_feat[target_col]

print(f"X_train shape: {X_train.shape} | Y_train.shape: {Y_train.shape}")
print(f"X_val_shape: {X_val.shape} | Y_val.shape: {Y_val.shape}")
print(f"X_test shape: {X_test.shape} | Y_test.shape: {Y_test.shape}")


X_train shape: (3886017, 12) | Y_train.shape: (3886017,)
X_val_shape: (896971, 12) | Y_val.shape: (896971,)
X_test shape: (295357, 12) | Y_test.shape: (295357,)


## Preprocessing


For this first baseline:
- numeric columns will be imputed and scaled
- categorical columns will be imputed and one-hot encoded

I know this is not needed for this dataset, since there are missing or null values, following this step because it is good practice to do

Because the classes are extremely imbalanced, I will start with `class_weight="balanced"` for the baseline.

In [15]:
numeric_transformer =  Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [16]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("OneHotEncoder",OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numeric_transformer,numeric_cols),
        ('cat',categorical_transformer,categorical_cols)
    ]
)

# Baseline Model

Learn what is saga \
Why is logistic regression called iteration optimization algorithm problem? \
Why logistic regression? \
What are the other better baseline models


In [18]:
baseline_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=300,
            solver="saga",
            n_jobs=1,
            random_state=42,
            class_weight="balanced",
        )),
    ]
)

## Train model

In [19]:
baseline_model.fit(X_train,Y_train)
print("Baseline Model Training Completed")

c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\fraud\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Baseline Model Training Completed


c:\Users\surab\OneDrive\Documents\Personal\Projects\Fraud_Detection\fraud\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# Score validation and test sets

The model outputs probabilities.  
I will use those probabilities first, then decide the classification threshold using the validation set.

In [22]:
val_scores = baseline_model.predict_proba(X_val)[:, 1]
test_scores = baseline_model.predict_proba(X_test)[:, 1]

print("Validation score range:", val_scores.min(), "to", val_scores.max())
print("Test score range      :", test_scores.min(), "to", test_scores.max())

Validation score range: 3.981437134963144e-06 to 0.983671728294677
Test score range      : 1.4140284922941621e-05 to 0.9864666199610276


Lower bound(~0) for Validation score range: 0.000004 - Model is extremely confident some samples are not fraud \
Upper bound(~1) for Validation score range: 0.9837- Model is very confident some samples are fraud

Similar for Test data as well

In [ ]:
# ROC-AUC helps us understand how well the model ranks fraud vs non-fraud transactions across different thresholds. 
# ROC-AUC full form is Receiver Operating Characteristic - Area Under the Curve. Why?
val_roc_auc = roc_auc_score(Y_val, val_scores) 

#PR-AUC (Precision - Recall Area under the curve) focuses on "When the model predicts a fraud, how often is it correct?"
val_pr_auc = average_precision_score(Y_val, val_scores)

test_roc_auc = roc_auc_score(Y_test, test_scores)
test_pr_auc = average_precision_score(Y_test, test_scores)

print("Validation ROC-AUC:", round(val_roc_auc, 6))
print("Validation PR-AUC :", round(val_pr_auc, 6))
print()
print("Test ROC-AUC      :", round(test_roc_auc, 6))
print("Test PR-AUC       :", round(test_pr_auc, 6))

Validation ROC-AUC: 0.902162
Validation PR-AUC : 0.014184

Test ROC-AUC      : 0.941116
Test PR-AUC       : 0.048853


We see that we have good ROC-AUC score but very low PR-AUC score.

## Choosing threshold using validation data

For the first baseline, I will choose the threshold that gives the best validation F1 score.

In [26]:
precisions, recalls, thresholds = precision_recall_curve(Y_val, val_scores)

In [ ]:
threshold_results = []
for t in thresholds:
    val_pred = (val_scores >= t).astype(int)

    threshold_results.append({
        "threshold": t,
        "precision": precision_score(Y_val, val_pred, zero_division=0),
        "recall": recall_score(Y_val, val_pred, zero_division=0),
        "f1": f1_score(Y_val, val_pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results).sort_values("f1", ascending=False).reset_index(drop=True)
threshold_df.head(10)

In [ ]:
best_threshold = threshold_df.loc[0, "threshold"]

print("Chosen threshold from validation:", round(float(best_threshold), 6))
print(threshold_df.head(5))